# CCBNet Experiments Reproduction (Google Colab)

This notebook **uses the upstream CCBNet repository directly** (`AbeleMM/ccbnet`) and runs the experiment matrix used in the paper.

- No local code modifications are required in the upstream repo.
- Results are saved under `/content/ccbnet/results_colab`.


## 0) Runtime setup
Use **Runtime → Change runtime type** and select a high-RAM CPU runtime if available.

In [ ]:
%cd /content
!rm -rf ccbnet
!git clone https://github.com/AbeleMM/ccbnet.git
%cd /content/ccbnet
!git rev-parse --short HEAD
!python --version

In [ ]:
# Dependency install with robust fallbacks for Colab/Poetry version differences
import os, subprocess, importlib

def sh(cmd, check=True):
    print('>>', cmd)
    proc = subprocess.run(cmd, shell=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {cmd}')
    return proc.returncode

installed = False

if os.path.exists('pyproject.toml'):
    # Poetry 2 often requires the export plugin explicitly
    sh('pip install -q poetry poetry-plugin-export')

    export_ok = (
        sh('poetry export -f requirements.txt --without-hashes -o /tmp/ccbnet-requirements.txt', check=False) == 0
        or sh('poetry self add poetry-plugin-export && poetry export -f requirements.txt --without-hashes -o /tmp/ccbnet-requirements.txt', check=False) == 0
    )

    if export_ok:
        sh('pip install -q -r /tmp/ccbnet-requirements.txt')
        installed = True
    else:
        print('Poetry export failed; trying pip install from project metadata...')
        if sh('pip install -q .', check=False) == 0:
            installed = True

if not installed:
    print('Falling back to pinned packages from the paper setup...')
    sh('pip install -q pgmpy==0.1.24 openmined-psi==2.0.2 tenseal==0.3.14 numpy pandas networkx scipy matplotlib seaborn tqdm')

for mod in ['pgmpy','numpy','pandas','networkx','matplotlib','seaborn','tqdm']:
    m = importlib.import_module(mod)
    print(mod, getattr(m, '__version__', 'n/a'))


In [ ]:
# Quick repo orientation
!find . -maxdepth 2 -type f | sed 's#^./##' | sort | head -n 200

In [ ]:
# Discover command line options
!python -m ccbnet.run --help || python ccbnet/run.py --help || true

## 1) Experiment configuration
The grid below mirrors the paper settings and can be tuned to fit Colab time constraints.

In [ ]:
from pathlib import Path
import itertools, json, os, subprocess, shlex, datetime

RESULTS_DIR = Path('/content/ccbnet/results_colab')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Core matrix from the paper
METHODS = ['CC', 'DOM', 'CCBNet', 'CCBNetJ']
NETWORKS_SMALL = ['ASIA', 'CHILD', 'ALARM', 'INSURANCE', 'WIN95PTS']
NETWORKS_LARGE = ['ANDES', 'PIGS', 'LINK', 'MUNIN2']
PARTIES = [2, 4, 8]
OVERLAPS = [0.1, 0.3, 0.5]
SPLITS = ['related', 'random']
SEED = 12345

# Set to True for quick smoke run
SMOKE = False
if SMOKE:
    METHODS = ['CCBNet']
    NETWORKS_SMALL = ['ASIA']
    PARTIES = [2]
    OVERLAPS = [0.3]
    SPLITS = ['related']

cfg = {
    'methods': METHODS,
    'networks_small': NETWORKS_SMALL,
    'networks_large': NETWORKS_LARGE,
    'parties': PARTIES,
    'overlaps': OVERLAPS,
    'splits': SPLITS,
    'seed': SEED,
    'timestamp_utc': datetime.datetime.utcnow().isoformat()
}
(RESULTS_DIR / 'run_config.json').write_text(json.dumps(cfg, indent=2))
cfg

## 2) Run experiments
This runner first tries `python -m ccbnet.run` and falls back to `python ccbnet/run.py`.

⚠️ Because CLI flags may differ by upstream version, adjust `build_cmd(...)` after checking `--help`.

In [ ]:
import csv, time

def build_cmd(method, network, parties, overlap, split, out_path):
    # Update flag names to match current upstream CLI after inspecting --help output.
    args = [
        '--method', method,
        '--network', network,
        '--parties', str(parties),
        '--overlap', str(overlap),
        '--split', split,
        '--seed', str(SEED),
        '--out', str(out_path),
    ]
    return [
        ['python', '-m', 'ccbnet.run', *args],
        ['python', 'ccbnet/run.py', *args],
    ]

def run_one(cmd_candidates):
    for cmd in cmd_candidates:
        try:
            print('>>', ' '.join(shlex.quote(x) for x in cmd))
            subprocess.check_call(cmd)
            return True
        except subprocess.CalledProcessError:
            continue
    return False

rows = []
jobs = list(itertools.product(METHODS, NETWORKS_SMALL, PARTIES, OVERLAPS, SPLITS))
print('Total jobs:', len(jobs))

for i, (method, network, parties, overlap, split) in enumerate(jobs, 1):
    run_id = f'{method}__{network}__p{parties}__o{overlap}__{split}'
    out_path = RESULTS_DIR / f'{run_id}.json'
    start = time.time()
    ok = run_one(build_cmd(method, network, parties, overlap, split, out_path))
    elapsed = time.time() - start
    rows.append({
        'run_id': run_id, 'method': method, 'network': network, 'parties': parties,
        'overlap': overlap, 'split': split, 'ok': ok, 'elapsed_sec': round(elapsed, 3),
        'out_path': str(out_path),
    })
    print(f'[{i}/{len(jobs)}] {run_id}:', 'OK' if ok else 'FAILED')

with open(RESULTS_DIR / 'run_log.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print('Saved run log to', RESULTS_DIR / 'run_log.csv')

## 3) Aggregate metrics and recreate paper-style plots
Expected metrics per run: Brier score, computation overhead, communication values.

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

records = []
for p in RESULTS_DIR.glob('*.json'):
    try:
        d = json.loads(p.read_text())
    except Exception:
        continue
    # Adapt key names to upstream output schema if needed
    records.append({
        'file': p.name,
        'method': d.get('method'),
        'network': d.get('network'),
        'parties': d.get('parties'),
        'overlap': d.get('overlap'),
        'split': d.get('split'),
        'brier': d.get('brier_score', d.get('brier')),
        'comp_overhead': d.get('computation_overhead', d.get('comp_overhead')),
        'comm_values': d.get('communicated_values', d.get('comm_values')),
    })

df = pd.DataFrame(records)
display(df.head())
print('Rows:', len(df))

if len(df):
    df.to_csv(RESULTS_DIR / 'aggregated_metrics.csv', index=False)
    print('Saved:', RESULTS_DIR / 'aggregated_metrics.csv')

    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    sns.barplot(data=df, x='network', y='brier', hue='method', ax=axes[0])
    axes[0].set_title('Brier Score (lower is better)')
    axes[0].tick_params(axis='x', rotation=45)

    sns.barplot(data=df, x='network', y='comp_overhead', hue='method', ax=axes[1])
    axes[1].set_title('Computation Overhead')
    axes[1].tick_params(axis='x', rotation=45)

    sns.barplot(data=df, x='network', y='comm_values', hue='method', ax=axes[2])
    axes[2].set_title('Communicated Values')
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plot_path = RESULTS_DIR / 'paper_style_summary.png'
    plt.savefig(plot_path, dpi=160)
    print('Saved:', plot_path)


In [ ]:
# Export all artifacts for download
%cd /content
!zip -r ccbnet_colab_artifacts.zip ccbnet/results_colab > /dev/null
from google.colab import files
files.download('/content/ccbnet_colab_artifacts.zip')